In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install --quiet anndata scanpy h5py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

In [4]:
# =============================================================================
# VivOME, prototype export
#
# Produces the artifacts the application needs that the v3 export does not
# already contain. Nothing here retrains anything. The frozen reference is
# loaded and used exactly as exported.
#
# Sections A to E need only the v3 export plus the proteomics TSVs, and run
# in well under a minute. Section F is the only one that loads the RNA h5ad,
# it is slow and memory heavy, and it is the only one that can be skipped.
# =============================================================================

import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")

DRIVE  = "/content/drive/My Drive/Vivome - Live Atlas"
V3_EXP = f"{DRIVE}/Data/Results/ReferenceProjection_v3/export"
PROT   = f"{DRIVE}/Data/scProteomics"
APP    = f"{DRIVE}/Data/Results/ReferenceProjection_v3/app_export"
os.makedirs(APP, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOOTH_K, SMOOTH_ALPHA, SMOOTH_PCA = 15, 0.60, 50

PROV = json.load(open(f"{V3_EXP}/provenance.json"))
FS   = pd.read_csv(f"{V3_EXP}/feature_space_genes.csv")["gene"].astype(str).str.upper().tolist()
meta = pd.read_csv(f"{V3_EXP}/reference_metadata.csv")

# Derive the class mapping from the metadata itself rather than assuming
# alphabetical order, then check that the assumption held anyway.
pairs   = meta[["class_idx", "class_name"]].drop_duplicates().sort_values("class_idx")
classes = pairs["class_name"].astype(str).tolist()
assert list(pairs["class_idx"]) == list(range(len(classes))), "class_idx is not 0..n-1"
assert classes == sorted(classes), "class_idx order is not alphabetical, downstream code assumed it is"
cls2idx = {c: i for i, c in enumerate(classes)}
MAC, MON = cls2idx.get("macrophage", -1), cls2idx.get("monocyte", -1)
SUPPORTED = [i for i in (MAC, MON) if i >= 0]

Z_ref_all = np.load(f"{V3_EXP}/reference_embedding.npy")
CENTROIDS = np.load(f"{V3_EXP}/reference_centroids.npy")
assert np.allclose(np.linalg.norm(CENTROIDS, axis=1), 1.0, atol=1e-3)
assert len(meta) == len(Z_ref_all), "metadata and embedding row counts disagree"

print(f"v3 export: {len(FS)} genes, {len(classes)} classes, seed {PROV['production_seed']}")
print(f"reference embedding {Z_ref_all.shape}, centroids {CENTROIDS.shape}")
print(f"cross modal supported classes: {[classes[i] for i in SUPPORTED]}")
print(f"device {DEVICE}")
print(f"writing to {APP}")

v3 export: 9002 genes, 22 classes, seed 0
reference embedding (85233, 128), centroids (22, 128)
cross modal supported classes: ['macrophage', 'monocyte']
device cuda
writing to /content/drive/My Drive/Vivome - Live Atlas/Data/Results/ReferenceProjection_v3/app_export


In [5]:
# -----------------------------------------------------------------------------
# The frozen reference, loaded exactly as the cell line comparison loaded it
# -----------------------------------------------------------------------------

def mlp(in_dim, hidden, dropout):
    layers, last = [], in_dim
    for h in hidden:
        layers += [nn.Linear(last, h, bias=False), nn.LayerNorm(h), nn.GELU()]
        if dropout: layers.append(nn.Dropout(dropout))
        last = h
    return nn.Sequential(*layers)

class ModulePoolEnc(nn.Module):
    def __init__(self, G, hidden, dropout, latent_dim, assign):
        super().__init__()
        self.register_buffer("A", assign)
        K = assign.shape[1]
        self.body = mlp(2*G + 2*K, hidden, dropout)
        self.proj = nn.Linear(hidden[-1], latent_dim, bias=False)
        self.norm = nn.LayerNorm(latent_dim)
    def forward(self, x, m):
        num  = (x*m) @ self.A
        den  = m @ self.A
        pool = num / (den + 1e-6)
        cov  = den / (self.A.sum(0, keepdim=True) + 1e-6)
        z = self.norm(self.proj(self.body(torch.cat([x*m, m, pool, cov], 1))))
        return F.normalize(z, dim=-1)

class ClassifierHead(nn.Module):
    def __init__(self, latent_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(latent_dim, n_classes, bias=True)
    def forward(self, z): return self.fc(z)

class ReferenceModel(nn.Module):
    def __init__(self, in_dim, n_classes, hidden, dropout, latent_dim, assign):
        super().__init__()
        self.encoder    = ModulePoolEnc(in_dim, hidden, dropout, latent_dim, assign)
        self.classifier = ClassifierHead(latent_dim, n_classes)
        self.latent_dim = latent_dim
    def forward(self, x, m):
        z = self.encoder(x, m)
        return z, self.classifier(z)

mod    = np.load(f"{V3_EXP}/module_assignment.npy")
K_MOD  = int(mod.max() + 1)
ASSIGN = torch.zeros(len(FS), K_MOD)
ASSIGN[torch.arange(len(FS)), torch.from_numpy(mod.astype(np.int64))] = 1.0
ASSIGN = ASSIGN.to(DEVICE)

REF_MODEL = ReferenceModel(len(FS), len(classes), (1024, 512), 0.1,
                           PROV["latent_dim"], ASSIGN).to(DEVICE)
REF_MODEL.load_state_dict(torch.load(f"{V3_EXP}/reference_model.pt", map_location=DEVICE),
                          strict=True)
REF_MODEL.eval()
print(f"reference loaded, {sum(p.numel() for p in REF_MODEL.parameters()):,} params")


def zscore_cols(df, eps=1e-8):
    return (df - df.mean(axis=0)) / (df.std(axis=0, ddof=0) + eps)

def clean_numeric(df):
    df = df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return df.apply(lambda c: c.fillna(c.median()), axis=0).fillna(0.0)

def collapse_dup_cols(df):
    return df.T.groupby(level=0).median().T if df.columns.duplicated().any() else df

def align_to_feature_space(gene_values):
    have = [g for g in FS if g in set(gene_values.columns)]
    Xa = pd.DataFrame(0.0, index=gene_values.index, columns=FS, dtype=np.float32)
    Xa[have] = gene_values[have].values
    Ma = np.zeros((len(gene_values), len(FS)), dtype=np.float32)
    Ma[:, [FS.index(g) for g in have]] = 1.0
    return Xa.values.astype(np.float32), Ma, len(have)

def fuzzy_smooth(X_shared_in, X_full_own, k=SMOOTH_K, alpha=SMOOTH_ALPHA,
                 n_pca=SMOOTH_PCA, seed=0, chunk=2000):
    n = X_full_own.shape[0]
    k = int(min(k, max(2, n-1)))
    comp = int(min(n_pca, X_full_own.shape[1], max(2, n-1)))
    P = PCA(n_components=comp, random_state=seed).fit_transform(
        np.asarray(X_full_own, dtype=np.float32))
    P /= (np.linalg.norm(P, axis=1, keepdims=True) + 1e-8)
    nn_ = NearestNeighbors(n_neighbors=k+1, metric="cosine").fit(P)
    dist, idx = nn_.kneighbors(P)
    idx, dist = idx[:, 1:], dist[:, 1:]
    w = np.exp(-(dist**2) / (dist[:, -1:]**2 + 1e-8))
    w = (w / (w.sum(1, keepdims=True) + 1e-8)).astype(np.float32)
    out = np.empty_like(X_shared_in, dtype=np.float32)
    for i in range(0, n, chunk):
        sl = slice(i, min(i+chunk, n))
        out[sl] = (1-alpha)*X_shared_in[sl] + alpha*np.einsum(
            "ijk,ij->ik", X_shared_in[idx[sl]], w[sl], optimize=True)
    return out

@torch.no_grad()
def encode(model, X, M, bs=2048):
    model.eval()
    Z, Xt, Mt = [], torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)), \
                   torch.from_numpy(np.ascontiguousarray(M, dtype=np.float32))
    for i in range(0, len(Xt), bs):
        z, _ = model(Xt[i:i+bs].to(DEVICE), Mt[i:i+bs].to(DEVICE))
        Z.append(z.cpu())
    return torch.cat(Z).numpy()

def load_proteomics(path, sep="\t"):
    df = pd.read_csv(path, sep=sep, index_col=0)
    df.columns = df.columns.astype(str).str.upper()
    df = collapse_dup_cols(df)
    dz = zscore_cols(clean_numeric(df)).astype(np.float32)
    Xr, M, n_have = align_to_feature_space(dz)
    return fuzzy_smooth(Xr, dz.values.astype(np.float32)), M, n_have, dz.index

print("helpers defined")

reference loaded, 20,058,134 params
helpers defined


In [6]:
# =============================================================================
# SECTION A, plotting coordinates for the atlas viewer
#
# The site currently plots the old 2,903 gene build. These are the v3
# equivalents. The fitted PCA is saved so that every future dataset lands in
# the same three dimensional projection rather than its own.
# =============================================================================

pca3 = PCA(n_components=3, random_state=0).fit(Z_ref_all)
PC_ref = pca3.transform(Z_ref_all)

rna_meta = meta.copy()
rna_meta["PC1"], rna_meta["PC2"], rna_meta["PC3"] = PC_ref[:,0], PC_ref[:,1], PC_ref[:,2]
rna_meta["latent_dim"] = PROV["latent_dim"]
rna_meta["modality"] = "RNA"
rna_meta.to_csv(f"{APP}/atlas_RNA_v3_meta.csv", index=False)

np.savez(f"{APP}/pca3_projection.npz",
         components=pca3.components_.astype(np.float32),
         mean=pca3.mean_.astype(np.float32),
         explained_variance_ratio=pca3.explained_variance_ratio_.astype(np.float32))

print(f"RNA plotting coords written, {len(rna_meta):,} cells")
print(f"explained variance by 3 components: {pca3.explained_variance_ratio_.sum()*100:.1f}%")
print(f"  per component: {np.round(pca3.explained_variance_ratio_*100, 2)}")

RNA plotting coords written, 85,233 cells
explained variance by 3 components: 69.7%
  per component: [38.05 19.12 12.55]


In [7]:
# =============================================================================
# SECTION B, the protein side, embedded and projected into the same space
# =============================================================================

Xp, Mp, n_have, prot_index = load_proteomics(f"{PROT}/blood_joint_cells_by_proteins_GENELEVEL.tsv")
cov_scope2 = 100 * Mp[0].mean()

raw = pd.read_csv(f"{PROT}/blood_joint_cells_by_proteins.tsv", sep="\t")
y_scope2 = np.array([cls2idx.get(str(t), -1) for t in raw["cell_type"].astype(str)])
assert len(y_scope2) == len(Xp), "label rows and matrix rows disagree"

Z_prot   = encode(REF_MODEL, Xp, Mp)
s_prot   = Z_prot @ CENTROIDS.T
pred     = s_prot.argmax(1)
PC_prot  = pca3.transform(Z_prot)

# max cosine to any single reference cell, the validated abstention score
CH = 256
max_cos = np.concatenate([ (Z_prot[i:i+CH] @ Z_ref_all.T).max(1)
                           for i in range(0, len(Z_prot), CH) ])
abstained = max_cos < PROV["abstain_threshold"]

prot_meta = pd.DataFrame({
    "cell": np.arange(len(Z_prot)),
    "cell_id": prot_index.astype(str),
    "modality": "PROT",
    "latent_dim": PROV["latent_dim"],
    "pred_class_idx": pred,
    "pred_class_name": [classes[c] for c in pred],
    "true_class_name": [classes[c] if c >= 0 else "unknown" for c in y_scope2],
    "max_cos_ref": max_cos,
    "abstained": abstained,
    "PC1": PC_prot[:,0], "PC2": PC_prot[:,1], "PC3": PC_prot[:,2],
})
prot_meta.to_csv(f"{APP}/atlas_PROT_v3_meta.csv", index=False)
np.save(f"{APP}/prot_embedding_scope2.npy", Z_prot.astype(np.float32))

print(f"SCoPE2: {len(Z_prot)} cells, {n_have} of {len(FS)} genes ({cov_scope2:.1f}% coverage)")
print(f"abstained at threshold {PROV['abstain_threshold']:.4f}: {100*abstained.mean():.1f}%")
print(f"assigned into a supported class: {100*np.isin(pred, SUPPORTED).mean():.1f}%")

SCoPE2: 1490 cells, 2907 of 9002 genes (32.3% coverage)
abstained at threshold 0.9779: 80.2%
assigned into a supported class: 45.8%


In [8]:
# =============================================================================
# SECTION C, latent_centroid_cosine
#
# Currently a pending metric in the manifest, blocked on 128 dimensional
# latent coordinates. Those now exist, so it can be measured for the classes
# that have cross modal coverage. Classes without protein cells stay pending,
# which is the correct record, not a gap.
# =============================================================================

rows = []
for ci, cname in enumerate(classes):
    m = (y_scope2 == ci)
    if m.sum() < 2:
        rows.append({"class_idx": ci, "class_name": cname, "n_prot_cells": int(m.sum()),
                     "latent_centroid_cosine": None, "status": "pending",
                     "reason": "no cross modal coverage"})
        continue
    c_prot = Z_prot[m].mean(0)
    c_prot = c_prot / (np.linalg.norm(c_prot) + 1e-12)
    rows.append({"class_idx": ci, "class_name": cname, "n_prot_cells": int(m.sum()),
                 "latent_centroid_cosine": float(c_prot @ CENTROIDS[ci]),
                 "status": "measured", "reason": "128-d latent, SCoPE2 true labels"})

LCC = pd.DataFrame(rows)
LCC.to_csv(f"{APP}/latent_centroid_cosine.csv", index=False)
meas = LCC[LCC.status == "measured"]
print(f"measured for {len(meas)} of {len(classes)} classes, the rest correctly pending")
print(meas[["class_name", "n_prot_cells", "latent_centroid_cosine"]].to_string(index=False))

measured for 2 of 22 classes, the rest correctly pending
class_name  n_prot_cells  latent_centroid_cosine
macrophage           394                0.189762
  monocyte          1096                0.830313


In [9]:
# =============================================================================
# SECTION D, modality probe
#
# Also pending in the manifest. If a linear probe recovers the source modality
# from the latent coordinates, then same type cells point the same way while
# the representations stay modality specific. The old build scored 98.7
# percent. RNA is subsampled to the protein count so the number is not an
# artefact of a 57 to 1 imbalance.
# =============================================================================

rng = np.random.default_rng(0)
sub = rng.choice(len(Z_ref_all), len(Z_prot), replace=False)
Xp_probe = np.vstack([Z_ref_all[sub], Z_prot])
yp_probe = np.r_[np.zeros(len(sub), int), np.ones(len(Z_prot), int)]

cv = StratifiedKFold(5, shuffle=True, random_state=0)
sc = cross_val_score(LogisticRegression(max_iter=2000, C=1.0),
                     Xp_probe, yp_probe, cv=cv, scoring="balanced_accuracy")

probe = {"modality_probe_balanced_accuracy_pct": float(100*sc.mean()),
         "std_pct": float(100*sc.std()),
         "folds": [float(100*s) for s in sc],
         "n_per_class": int(len(Z_prot)),
         "basis": "5-fold CV, logistic regression on 128-d latent, RNA subsampled to protein n"}
json.dump(probe, open(f"{APP}/modality_probe.json", "w"), indent=2)

print(f"modality probe balanced accuracy: {100*sc.mean():.2f}% +/- {100*sc.std():.2f}")
print("50% would mean the modalities are indistinguishable in latent space")

modality probe balanced accuracy: 98.99% +/- 0.28
50% would mean the modalities are indistinguishable in latent space


In [10]:
# =============================================================================
# SECTION E, does restricting to the supported label space help
#
# A product decision, not a methodology one. 22 way assignment lets 20 classes
# with zero protein evidence compete for the argmax. This measures what
# happens when only classes with cross modal support are candidates, which is
# what the support map and abstention design already imply the service should
# do.
# =============================================================================

valid = y_scope2 >= 0
res = []

res.append({"regime": "unrestricted, all 22 classes",
            "accuracy_pct": 100*(pred[valid] == y_scope2[valid]).mean(),
            "balanced_acc_pct": 100*balanced_accuracy_score(y_scope2[valid], pred[valid]),
            "n_candidate_classes": len(classes)})

if len(SUPPORTED) >= 2:
    pred_r = np.array(SUPPORTED)[s_prot[:, SUPPORTED].argmax(1)]
    res.append({"regime": "restricted to cross modal supported classes",
                "accuracy_pct": 100*(pred_r[valid] == y_scope2[valid]).mean(),
                "balanced_acc_pct": 100*balanced_accuracy_score(y_scope2[valid], pred_r[valid]),
                "n_candidate_classes": len(SUPPORTED)})

    margin = s_prot[:, MAC] - s_prot[:, MON]
    is_mac = (y_scope2 == MAC).astype(int)
    auc = roc_auc_score(is_mac[valid], margin[valid])
    grid = np.quantile(margin[valid], np.linspace(0.01, 0.99, 199))
    bals = [100*balanced_accuracy_score(is_mac[valid], (margin[valid] > t).astype(int)) for t in grid]
    best = int(np.argmax(bals))
    res.append({"regime": "restricted, threshold tuned on the margin",
                "accuracy_pct": None, "balanced_acc_pct": bals[best],
                "n_candidate_classes": len(SUPPORTED)})
    print(f"macrophage vs monocyte AUC: {auc:.4f}")
    print(f"best margin threshold {grid[best]:+.4f}, balanced accuracy {bals[best]:.2f}%")

SUP = pd.DataFrame(res)
SUP.to_csv(f"{APP}/support_restricted_assignment.csv", index=False)
print()
print(SUP.to_string(index=False))
print()
print("If the restricted row is far above the unrestricted one, the service")
print("should constrain candidates to the supported label space at serve time,")
print("and the gap is a measurement of distractor contamination, not of the")
print("modality gap itself.")

macrophage vs monocyte AUC: 0.9278
best margin threshold -0.1586, balanced accuracy 85.97%

                                     regime  accuracy_pct  balanced_acc_pct  n_candidate_classes
               unrestricted, all 22 classes     45.369128         31.083265                   22
restricted to cross modal supported classes     86.174497         79.791535                    2
  restricted, threshold tuned on the margin           NaN         85.966505                    2

If the restricted row is far above the unrestricted one, the service
should constrain candidates to the supported label space at serve time,
and the gap is a measurement of distractor contamination, not of the
modality gap itself.


In [11]:
# =============================================================================
# SECTION F, per reference cell property scores.  SLOW, OPTIONAL.
#
# The only cell that loads the RNA h5ad. It rebuilds PROP_MAT, which v3
# computed in memory and never exported, so property transfer currently
# cannot be served. Skip this cell if the prototype ships without property
# transfer, nothing else depends on it.
# =============================================================================

RUN_SECTION_F = True

if RUN_SECTION_F:
    import h5py, anndata as ad, scanpy as sc_
    RNA_H5AD = f"{DRIVE}/Data/scRNA-seq/Blood_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad"

    PROPERTY_SETS = {
        "cell_cycle_S":  ["MCM5","PCNA","TYMS","RRM1","RRM2","MCM2","MCM6","CDC6","GINS2"],
        "cell_cycle_G2M":["HMGB2","CDK1","TOP2A","MKI67","CCNB2","UBE2C","AURKA","BUB1"],
        "interferon":    ["ISG15","IFI6","IFI44L","MX1","OAS1","STAT1","IRF7","IFIT1","IFIT3"],
        "oxphos":        ["NDUFA1","NDUFB1","COX5A","COX7C","ATP5F1E","UQCRB","SDHB"],
        "glycolysis":    ["GAPDH","ENO1","PKM","LDHA","PGK1","TPI1","ALDOA"],
        "antigen_presentation": ["HLA-DRA","HLA-DRB1","HLA-DPA1","CD74","HLA-DMA"],
        "inflammatory":  ["IL1B","TNF","CXCL8","CCL3","CCL4","NFKBIA","SOD2"],
        "ribosome":      ["RPL13A","RPS6","RPL10","RPS3","RPL3","RPS18"],
    }

    with h5py.File(RNA_H5AD, "r") as f:
        var = f["var"]
        gnode = var[var.attrs["_index"]] if "_index" in var.attrs else var["_index"]
        full_rna_genes = pd.Index(np.array(gnode).astype(str)).str.upper()

    adata = sc_.read_h5ad(RNA_H5AD)
    adata.var_names = full_rna_genes
    adata.var_names_make_unique()

    # The property matrix is indexed by reference row. If the h5ad has been
    # filtered since v3 ran, the rows no longer correspond and the saved
    # matrix would be silently misaligned.
    assert adata.n_obs == len(meta), (
        f"h5ad has {adata.n_obs} cells, reference has {len(meta)}. "
        "Row correspondence is broken, do not save this matrix.")

    PROPS = {}
    for name, gset in PROPERTY_SETS.items():
        present = [g for g in gset if g in set(adata.var_names)]
        if len(present) < 3:
            print(f"[skip] {name}, only {len(present)} genes present"); continue
        sc_.tl.score_genes(adata, present, score_name=f"_s_{name}")
        PROPS[name] = adata.obs[f"_s_{name}"].values.astype(np.float32)
        print(f"[score] {name:<22s} genes={len(present):2d}")

    PROP_NAMES = list(PROPS.keys())
    PROP_MAT = np.stack([PROPS[k] for k in PROP_NAMES], 1).astype(np.float32)
    np.save(f"{APP}/reference_properties.npy", PROP_MAT)
    json.dump({"property_names": PROP_NAMES,
               "shape": list(PROP_MAT.shape),
               "row_order": "matches reference_embedding.npy and reference_metadata.csv",
               "note": "ship only properties that passed validation, see property_validation.csv"},
              open(f"{APP}/property_names.json", "w"), indent=2)
    del adata
    print(f"\nreference_properties.npy written, shape {PROP_MAT.shape}")
else:
    print("Section F skipped. The prototype will have no property transfer.")

[score] cell_cycle_S           genes= 9
[score] cell_cycle_G2M         genes= 8
[score] interferon             genes= 9
[score] oxphos                 genes= 7
[score] glycolysis             genes= 7
[score] antigen_presentation   genes= 5
[score] inflammatory           genes= 7
[score] ribosome               genes= 6

reference_properties.npy written, shape (85233, 8)


In [12]:
# =============================================================================
# Summary and gate
# =============================================================================

summary = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "source_export": V3_EXP,
    "gene_list_hash": PROV["gene_list_hash"],
    "n_genes": len(FS), "n_classes": len(classes), "latent_dim": PROV["latent_dim"],
    "supported_classes": [classes[i] for i in SUPPORTED],
    "rna_cells": int(len(meta)), "prot_cells": int(len(Z_prot)),
    "scope2_coverage_pct": float(cov_scope2),
    "pca3_explained_variance_pct": float(pca3.explained_variance_ratio_.sum()*100),
    "abstain_threshold": PROV["abstain_threshold"],
    "abstain_rate_pct": float(100*abstained.mean()),
    "modality_probe_balanced_accuracy_pct": probe["modality_probe_balanced_accuracy_pct"],
    "section_F_run": bool(RUN_SECTION_F),
}
json.dump(summary, open(f"{APP}/app_export_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

expected = ["atlas_RNA_v3_meta.csv", "atlas_PROT_v3_meta.csv", "pca3_projection.npz",
            "prot_embedding_scope2.npy", "latent_centroid_cosine.csv",
            "modality_probe.json", "support_restricted_assignment.csv",
            "app_export_summary.json"]
if RUN_SECTION_F:
    expected += ["reference_properties.npy", "property_names.json"]

print(f"\n[files in {APP}]")
present = set(os.listdir(APP))
for f_ in expected:
    mb = os.path.getsize(f"{APP}/{f_}") / 1e6 if f_ in present else 0
    print(f"  {'OK  ' if f_ in present else 'MISS'} {f_:<38s} {mb:8.2f} MB")
missing = [f_ for f_ in expected if f_ not in present]
assert not missing, f"missing outputs: {missing}"
print("\n[gate] all expected outputs written. Download this folder.")

{
  "created": "2026-09-22 17:16:02",
  "source_export": "/content/drive/My Drive/Vivome - Live Atlas/Data/Results/ReferenceProjection_v3/export",
  "gene_list_hash": "5751dd1e569f",
  "n_genes": 9002,
  "n_classes": 22,
  "latent_dim": 128,
  "supported_classes": [
    "macrophage",
    "monocyte"
  ],
  "rna_cells": 85233,
  "prot_cells": 1490,
  "scope2_coverage_pct": 32.292823791503906,
  "pca3_explained_variance_pct": 69.72093200683594,
  "abstain_threshold": 0.9779149889945984,
  "abstain_rate_pct": 80.20134228187919,
  "modality_probe_balanced_accuracy_pct": 98.99328859060402,
  "section_F_run": true
}

[files in /content/drive/My Drive/Vivome - Live Atlas/Data/Results/ReferenceProjection_v3/app_export]
  OK   atlas_RNA_v3_meta.csv                      6.32 MB
  OK   atlas_PROT_v3_meta.csv                     0.15 MB
  OK   pca3_projection.npz                        0.00 MB
  OK   prot_embedding_scope2.npy                  0.76 MB
  OK   latent_centroid_cosine.csv               

In [13]:
# =============================================================================
# BUNDLE, everything the prototype build needs, in one zip
# =============================================================================

import os, json, time, hashlib, shutil, zipfile

BUNDLE_DIR  = f"{DRIVE}/_prototype_bundle"
STAGE       = "/content/_bundle_stage"          # local disk, fast, then copied
TABLES      = f"{DRIVE}/Data/Results/ReferenceProjection_v3/tables"
INCLUDE_DEMO_DATA = False   # tier 2 already has SCoPE2 embedded, so usually not needed

os.makedirs(BUNDLE_DIR, exist_ok=True)
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)

TIER1 = ["reference_model.pt", "feature_space_genes.csv", "module_assignment.npy",
         "reference_centroids.npy", "reference_embedding.npy",
         "reference_metadata.csv", "provenance.json"]
TIER2 = ["atlas_RNA_v3_meta.csv", "atlas_PROT_v3_meta.csv", "pca3_projection.npz",
         "prot_embedding_scope2.npy", "latent_centroid_cosine.csv",
         "modality_probe.json", "support_restricted_assignment.csv",
         "reference_properties.npy", "property_names.json",
         "app_export_summary.json"]

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

plan, missing = [], []
for name in TIER1:
    p = f"{V3_EXP}/{name}"
    (plan if os.path.exists(p) else missing).append((p, f"export/{name}") if os.path.exists(p) else p)
for name in TIER2:
    p = f"{APP}/{name}"
    (plan if os.path.exists(p) else missing).append((p, f"app_export/{name}") if os.path.exists(p) else p)
if os.path.isdir(TABLES):
    for name in sorted(os.listdir(TABLES)):
        if name.endswith((".csv", ".json")):
            plan.append((f"{TABLES}/{name}", f"tables/{name}"))
if INCLUDE_DEMO_DATA:
    for p in [f"{PROT}/blood_joint_cells_by_proteins_GENELEVEL.tsv",
              f"{PROT}/blood_joint_cells_by_proteins.tsv"]:
        if os.path.exists(p):
            plan.append((p, f"demo/{os.path.basename(p)}"))

assert not missing, f"required files absent, nothing zipped: {missing}"

# Manifest first, so it travels inside the archive it describes.
entries = []
for src, arc in plan:
    entries.append({"path": arc, "bytes": os.path.getsize(src), "sha256": sha256(src)})

manifest = {
    "bundle_created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "source_drive_root": DRIVE,
    "gene_list_hash": PROV["gene_list_hash"],
    "n_genes": len(FS), "n_classes": len(classes), "latent_dim": PROV["latent_dim"],
    "supported_classes": [classes[i] for i in SUPPORTED],
    "n_files": len(entries),
    "total_bytes": sum(e["bytes"] for e in entries),
    "files": entries,
}
mpath = f"{STAGE}/BUNDLE_MANIFEST.json"
json.dump(manifest, open(mpath, "w"), indent=2)

readme = f"""VivOME prototype bundle
Created {manifest['bundle_created']}
Gene list hash {manifest['gene_list_hash']}, {manifest['n_genes']} genes, {manifest['n_classes']} classes

  export/      frozen reference, the serving core. All 7 files required.
  app_export/  plotting coords, protein embedding, diagnostics, properties.
  tables/      measured numbers the site displays.

Verify with BUNDLE_MANIFEST.json, it carries a sha256 for every file.
Place VivOME_Prototype_Context.md and VivOME_Prototype_Download_Manifest.md
alongside this folder before starting the build.
"""
rpath = f"{STAGE}/README.txt"
open(rpath, "w").write(readme)

stamp = time.strftime("%Y%m%d_%H%M")
local_zip = f"{STAGE}/vivome_prototype_bundle_{stamp}.zip"

print(f"zipping {len(plan)} files, {manifest['total_bytes']/1e6:.1f} MB uncompressed")
with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    z.write(mpath, "vivome_prototype_bundle/BUNDLE_MANIFEST.json")
    z.write(rpath, "vivome_prototype_bundle/README.txt")
    for i, (src, arc) in enumerate(plan, 1):
        z.write(src, f"vivome_prototype_bundle/{arc}")
        print(f"  [{i:2d}/{len(plan)}] {arc:<46s} {os.path.getsize(src)/1e6:7.2f} MB")

# Built locally for speed, then copied to Drive in one write.
final_zip = f"{BUNDLE_DIR}/{os.path.basename(local_zip)}"
shutil.copy2(local_zip, final_zip)

with zipfile.ZipFile(final_zip) as z:
    assert z.testzip() is None, "archive failed its own integrity check"
    n_in = len(z.namelist())

print(f"\nzipped   {os.path.getsize(final_zip)/1e6:.1f} MB, {n_in} entries, integrity verified")
print(f"Drive    {final_zip}")
print(f"local    {local_zip}")
print("\nDownload from the Drive folder '_prototype_bundle', or uncomment below.")
# from google.colab import files; files.download(local_zip)

zipping 24 files, 154.8 MB uncompressed
  [ 1/24] export/reference_model.pt                        98.28 MB
  [ 2/24] export/feature_space_genes.csv                    0.06 MB
  [ 3/24] export/module_assignment.npy                      0.04 MB
  [ 4/24] export/reference_centroids.npy                    0.01 MB
  [ 5/24] export/reference_embedding.npy                   43.64 MB
  [ 6/24] export/reference_metadata.csv                     2.73 MB
  [ 7/24] export/provenance.json                            0.00 MB
  [ 8/24] app_export/atlas_RNA_v3_meta.csv                  6.32 MB
  [ 9/24] app_export/atlas_PROT_v3_meta.csv                 0.15 MB
  [10/24] app_export/pca3_projection.npz                    0.00 MB
  [11/24] app_export/prot_embedding_scope2.npy              0.76 MB
  [12/24] app_export/latent_centroid_cosine.csv             0.00 MB
  [13/24] app_export/modality_probe.json                    0.00 MB
  [14/24] app_export/support_restricted_assignment.csv      0.00 MB
  [15/24